# Bab 2: Pre-Model Workflow and Data Preprocessing

Notebook ini merangkum Bab 2 buku *scikit-learn Cookbook* (Third Edition) karya John Sukup. Bab ini membahas kualitas data sebagai faktor paling menentukan performa model: menangani missing data, scaling, encoding variabel kategorikal, menyusun pipeline, dan feature engineering, semuanya memakai tools bawaan scikit-learn.

Semua kode di notebook ini sudah dijalankan, jadi outputnya langsung terlihat begitu file dibuka.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.precision", 4)
np.random.seed(2024)

## 1. Dampak Kualitas Data Mentah terhadap Performa Model

Model machine learning belajar pola dari data, jadi kalau datanya bermasalah (missing values, outlier, fitur yang tidak relevan), kemampuan model untuk generalisasi ke data baru ikut menurun. Beberapa masalah data yang paling umum ditemui:

- **Missing data**: bisa muncul karena error saat pengumpulan data, kegagalan sistem, atau korupsi data.
- **Outlier**: bisa mendistorsi analisis statistik dan menyesatkan hasil model.
- **Variabel kategorikal**: kebanyakan algoritma ML butuh input numerik, jadi variabel kategorikal perlu diubah dulu.
- **Skala fitur yang berbeda-beda**: fitur dengan rentang nilai jauh berbeda bisa membuat model yang sensitif terhadap jarak (seperti KNN) berperforma buruk.
- **Data leakage**: terjadi ketika informasi dari luar data training "bocor" masuk ke proses pembuatan model, membuat metrik performa terlihat bagus padahal menyesatkan.

scikit-learn menyediakan tools untuk menangani semuanya: `Pipeline()` untuk merangkai langkah preprocessing dengan training, `ColumnTransformer()` untuk menerapkan transformasi berbeda ke kolom numerik dan kategorikal sekaligus, dan modul `sklearn.feature_selection` untuk memilih fitur yang paling relevan.

## 2. Menangani Missing Data

Kita akan membuat dataset mainan berisi data numerik acak dengan beberapa nilai yang sengaja dihilangkan, lalu mencoba tiga pendekatan imputasi yang disediakan scikit-learn: `SimpleImputer()`, `KNNImputer()`, dan `IterativeImputer()`.

In [2]:
np.random.seed(2024)  # untuk reproduktifitas
n_samples = 20
n_features = 10

# Buat data acak
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}
df = pd.DataFrame(data)

# Hilangkan sekitar 20% data secara acak di tiap kolom
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,NaN,42.0098,34.6804,49.9623,NaN,NaN,89.9546,41.1524,NaN
1,69.9109,9.5542,6.4364,31.2878,37.9665,NaN,82.0056,NaN,75.7976,91.3825
2,NaN,96.0910,59.6433,84.7104,NaN,84.1090,NaN,70.2881,1.7783,4.1177
3,NaN,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,NaN,NaN,80.0780
4,20.5019,NaN,89.2486,67.6559,58.6359,78.2257,60.9116,NaN,65.1142,99.1192
5,10.6063,76.8254,20.0527,5.3675,NaN,19.7031,34.4233,93.3995,72.2067,12.6403
6,72.7240,79.7923,50.2395,55.9214,6.1910,NaN,22.9669,21.0490,57.3585,14.3026
7,NaN,NaN,89.5382,69.4513,NaN,47.8856,NaN,33.6204,99.6857,6.6831
8,47.3846,NaN,25.5921,82.4197,73.4145,61.6637,29.1726,65.9467,61.0052,34.0527
9,44.8296,38.1651,NaN,31.1429,28.8655,NaN,41.0045,41.4603,50.2362,NaN


### SimpleImputer()

`SimpleImputer()` adalah pendekatan paling sederhana: mengganti nilai yang hilang dengan statistik tertentu, misalnya mean, median, atau nilai yang paling sering muncul. Cocok untuk fitur numerik.

In [3]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,53.8647,42.0098,34.6804,49.9623,57.8230,51.1450,89.9546,41.1524,48.4000
1,69.9109,9.5542,6.4364,31.2878,37.9665,57.8230,82.0056,50.7150,75.7976,91.3825
2,52.5586,96.0910,59.6433,84.7104,46.0559,84.1090,51.1450,70.2881,1.7783,4.1177
3,52.5586,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,50.7150,60.6167,80.0780
4,20.5019,53.8647,89.2486,67.6559,58.6359,78.2257,60.9116,50.7150,65.1142,99.1192
5,10.6063,76.8254,20.0527,5.3675,46.0559,19.7031,34.4233,93.3995,72.2067,12.6403
6,72.7240,79.7923,50.2395,55.9214,6.1910,57.8230,22.9669,21.0490,57.3585,14.3026
7,52.5586,53.8647,89.5382,69.4513,46.0559,47.8856,51.1450,33.6204,99.6857,6.6831
8,47.3846,53.8647,25.5921,82.4197,73.4145,61.6637,29.1726,65.9467,61.0052,34.0527
9,44.8296,38.1651,52.8975,31.1429,28.8655,57.8230,41.0045,41.4603,50.2362,48.4000


### KNNImputer()

`KNNImputer()` mengisi nilai yang hilang berdasarkan nilai dari tetangga terdekat di ruang fitur (ditentukan oleh `n_neighbors`). Pendekatan ini bisa menangkap hubungan antar fitur yang lebih halus dibanding `SimpleImputer()`.

In [4]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
knn_imputed_df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,48.9540,42.0098,34.6804,49.9623,93.9104,52.5493,89.9546,41.1524,59.8337
1,69.9109,9.5542,6.4364,31.2878,37.9665,86.7935,82.0056,45.4162,75.7976,91.3825
2,67.7523,96.0910,59.6433,84.7104,73.3383,84.1090,59.0129,70.2881,1.7783,4.1177
3,47.8809,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,64.5103,39.7268,80.0780
4,20.5019,31.6709,89.2486,67.6559,58.6359,78.2257,60.9116,25.9036,65.1142,99.1192
5,10.6063,76.8254,20.0527,5.3675,59.7168,19.7031,34.4233,93.3995,72.2067,12.6403
6,72.7240,79.7923,50.2395,55.9214,6.1910,40.4829,22.9669,21.0490,57.3585,14.3026
7,47.6297,77.8439,89.5382,69.4513,32.7533,47.8856,55.6834,33.6204,99.6857,6.6831
8,47.3846,35.1806,25.5921,82.4197,73.4145,61.6637,29.1726,65.9467,61.0052,34.0527
9,44.8296,38.1651,29.1883,31.1429,28.8655,59.6747,41.0045,41.4603,50.2362,35.0008


### IterativeImputer()

`IterativeImputer()` memodelkan tiap fitur yang punya missing value sebagai fungsi regresi dari fitur-fitur lainnya, dilakukan bergantian untuk semua fitur yang punya nilai hilang. Pendekatan ini lebih kompleks secara komputasi, tapi biasanya memberi hasil imputasi yang lebih akurat karena mempertimbangkan hubungan multivariat dalam data.

In [5]:
from sklearn.experimental import enable_iterative_imputer  # fitur eksperimental, wajib di-import dulu
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=2024)
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)
iterative_imputed_df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,54.3650,42.0098,34.6804,49.9623,58.6806,51.1781,89.9546,41.1524,48.2923
1,69.9109,9.5542,6.4364,31.2878,37.9665,60.0706,82.0056,50.7103,75.7976,91.3825
2,52.5397,96.0910,59.6433,84.7104,46.1213,84.1090,51.1554,70.2881,1.7783,4.1177
3,52.6306,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,50.6803,50.6509,80.0780
4,20.5019,53.1990,89.2486,67.6559,58.6359,78.2257,60.9116,50.6917,65.1142,99.1192
5,10.6063,76.8254,20.0527,5.3675,46.1563,19.7031,34.4233,93.3995,72.2067,12.6403
6,72.7240,79.7923,50.2395,55.9214,6.1910,55.9236,22.9669,21.0490,57.3585,14.3026
7,52.5581,53.9429,89.5382,69.4513,46.0705,47.8856,50.9698,33.6204,99.6857,6.6831
8,47.3846,54.2050,25.5921,82.4197,73.4145,61.6637,29.1726,65.9467,61.0052,34.0527
9,44.8296,38.1651,52.8826,31.1429,28.8655,55.9783,41.0045,41.4603,50.2362,48.4299


Pemilihan strategi imputasi tergantung beberapa faktor: tipe data, distribusi data, pola data yang hilang, dan seberapa banyak data yang hilang. Sebagai panduan kasar dari buku: kalau missing value di bawah 5%, `SimpleImputer()` sudah cukup karena ringan secara komputasi. Untuk 5-10% missing value, `KNNImputer()` cocok untuk data numerik murni, sementara `IterativeImputer()` cocok untuk data dengan tipe campuran. Di atas 10% missing value pada satu fitur saja, pertimbangkan untuk membuang fitur itu kecuali benar-benar penting.

Perlu diingat juga, tidak semua algoritma butuh imputasi. Decision tree dan Random Forest biasanya bisa menangani missing value secara native.

## 3. Teknik Scaling

Fitur dengan skala yang berbeda jauh (misalnya usia 0-100 vs pendapatan 0-100.000) bisa membuat algoritma yang sensitif terhadap jarak, seperti KNN, didominasi oleh fitur dengan skala besar. Kita akan memakai dataset hasil `IterativeImputer()` dari bagian sebelumnya untuk mencoba tiga teknik scaling.

Dua istilah yang sering tertukar tapi sebenarnya berbeda:

- **Standardization**: mengubah data supaya mean-nya 0 dan standar deviasinya 1.
- **Normalization**: mengubah rentang data supaya nilainya berada di antara 0 dan 1.

### StandardScaler()

Standardisasi mengubah fitur jadi punya mean 0 dan standar deviasi 1, dengan formula z = (x - mean) / std. Cocok dipakai kalau data cenderung berdistribusi normal.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)
scaled_df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.2719,0.0195,-0.3731,-0.8943,0.1888,0.0318,0.0019,1.6838,-0.7809,-0.0036
1,0.7560,-1.8749,-1.5922,-1.0439,-0.3917,0.0859,1.4269,-0.0005,0.6525,1.4291
2,-0.0009,1.7835,0.2313,1.3120,0.0029,1.0216,0.0009,0.8397,-2.4099,-1.4723
3,0.0030,-1.2145,1.0568,1.4580,-1.4118,1.5313,1.5513,-0.0018,-0.3879,1.0532
4,-1.3971,-0.0298,1.2459,0.5599,0.6085,0.7926,0.4518,-0.0013,0.2105,1.6863
5,-1.8283,0.9690,-1.1255,-2.1869,0.0046,-1.4853,-0.7726,1.8317,0.5039,-1.1889
6,0.8786,1.0945,-0.0910,0.0424,-1.9294,-0.0755,-1.3021,-1.2736,-0.1104,-1.1336
7,-0.0001,0.0016,1.2558,0.6391,0.0004,-0.3883,-0.0077,-0.7340,1.6408,-1.3870
8,-0.2256,0.0127,-0.9357,1.2109,1.3237,0.1480,-1.0153,0.6534,0.0405,-0.4770
9,-0.3369,-0.6654,-0.0004,-1.0503,-0.8322,-0.0733,-0.4684,-0.3975,-0.4051,0.0010


### MinMaxScaler()

Min-max scaling mengubah skala fitur ke rentang tetap, biasanya [0, 1], dengan formula x' = (x - x_min) / (x_max - x_min). Teknik ini menjaga hubungan relatif antar nilai sekaligus memastikan semua fitur berkontribusi setara.

In [7]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)
minmax_scaled_df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.6035,0.5178,0.3805,0.3546,0.5690,0.5559,0.4350,0.9628,0.4022,0.4650
1,0.7214,0.0000,0.0004,0.3136,0.4131,0.5719,0.8331,0.5386,0.7560,0.9186
2,0.5371,1.0000,0.5690,0.9599,0.5191,0.8490,0.4347,0.7502,0.0000,0.0000
3,0.5380,0.1805,0.8264,1.0000,0.1390,1.0000,0.8678,0.5383,0.4992,0.7996
4,0.1972,0.5043,0.8854,0.7536,0.6818,0.7812,0.5607,0.5384,0.6469,1.0000
5,0.0922,0.7774,0.1459,0.0000,0.5195,0.1066,0.2187,1.0000,0.7193,0.0897
6,0.7512,0.8117,0.4685,0.6116,0.0000,0.5241,0.0707,0.2180,0.5677,0.1072
7,0.5373,0.5129,0.8885,0.7753,0.5184,0.4315,0.4323,0.3539,1.0000,0.0270
8,0.4824,0.5160,0.2051,0.9322,0.8739,0.5903,0.1509,0.7033,0.6049,0.3151
9,0.4553,0.3306,0.4967,0.3118,0.2948,0.5247,0.3036,0.4386,0.4949,0.4664


### Normalizer()

Normalisasi (L1 atau L2) menskalakan tiap sampel (baris) supaya punya unit norm, bukan tiap kolom seperti dua scaler sebelumnya. Berguna untuk data sparse atau ketika kita ingin memperlakukan tiap sampel setara terlepas dari besarnya magnitude.

In [8]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer()
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)
normalized_df.head(10)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.3392,0.3136,0.2423,0.2000,0.2882,0.3385,0.2952,0.5189,0.2374,0.2786
1,0.3767,0.0515,0.0347,0.1686,0.2046,0.3237,0.4419,0.2732,0.4084,0.4924
2,0.2643,0.4834,0.3001,0.4262,0.2320,0.4232,0.2574,0.3536,0.0089,0.0207
3,0.2438,0.1166,0.3878,0.4077,0.0782,0.4502,0.3923,0.2347,0.2346,0.3709
4,0.0959,0.2489,0.4175,0.3165,0.2743,0.3659,0.2849,0.2371,0.3046,0.4637
5,0.0681,0.4934,0.1288,0.0345,0.2964,0.1265,0.2211,0.5998,0.4637,0.0812
6,0.4605,0.5053,0.3181,0.3541,0.0392,0.3541,0.1454,0.1333,0.3632,0.0906
7,0.2746,0.2818,0.4678,0.3628,0.2407,0.2502,0.2663,0.1756,0.5208,0.0349
8,0.2653,0.3035,0.1433,0.4614,0.4110,0.3452,0.1633,0.3692,0.3415,0.1906
9,0.3213,0.2735,0.3790,0.2232,0.2069,0.4012,0.2939,0.2971,0.3600,0.3471


Sama seperti imputasi, tidak semua data perlu di-scale. Model berbasis tree seperti decision tree dan Random Forest bekerja langsung dengan nilai mentah fitur, jadi scaling justru bisa berdampak buruk pada hasilnya. Sebaliknya, KNN dan SVM sangat sensitif terhadap skala, sementara linear regression bisa konvergen lebih cepat dengan fitur yang sudah distandarisasi.

## 4. Encoding Variabel Kategorikal

Variabel kategorikal terbagi dua: **nominal** (tanpa urutan, misalnya warna atau brand) dan **ordinal** (punya urutan jelas, misalnya rating 1-5). Pemilihan metode encoding tergantung jenis variabelnya.

In [9]:
np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame(
    {
        "Department": np.random.choice(categories, size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
        "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
    }
)
categorical_data.head(10)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


### OneHotEncoder()

One-hot encoding mengubah tiap kategori jadi kolom biner sendiri, sehingga model memperlakukan tiap kategori secara independen tanpa menganggapnya punya urutan. Kelemahannya, kalau fiturnya banyak kategori, hasilnya bisa jadi dataset yang sangat besar dan sparse (banyak nol).

In [10]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data, columns=onehot_encoder.get_feature_names_out()
)
onehot_encoded_df.head(10)

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


### LabelEncoder()

Label encoding memberi satu angka integer untuk tiap kategori. Cara ini sederhana dan efisien, tapi berisiko menyiratkan hubungan berurutan yang sebenarnya tidak ada (misalnya "Blue" = 2 tidak berarti dua kali lebih besar dari "Red" = 0). Perlu hati-hati menerapkannya hanya pada variabel yang benar-benar ordinal, atau yang modelnya tidak terpengaruh asumsi urutan (seperti tree-based model).

In [11]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()

for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(
        categorical_data[column]
    )

label_encoded_df.head(10)

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1
6,2,1,3
7,3,1,1
8,1,0,1
9,3,1,0


### ColumnTransformer()

Untuk dataset yang punya campuran fitur numerik dan kategorikal, `ColumnTransformer()` memungkinkan kita menerapkan transformasi berbeda ke kolom yang berbeda dalam satu langkah, misalnya scaling untuk kolom numerik dan one-hot encoding untuk kolom kategorikal, tanpa harus memprosesnya terpisah secara manual.

In [12]:
from sklearn.compose import ColumnTransformer

np.random.seed(2024)
mixed_data = pd.DataFrame(
    {
        "Age": np.random.randint(25, 65, size=20),
        "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
        "Experience": np.random.randint(1, 20, size=20),
        "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    }
)
mixed_data.head(10)

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


In [13]:
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

transformed_data = column_transformer.fit_transform(mixed_data)

numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = column_transformer.named_transformers_["cat"].get_feature_names_out(
    ["Department", "Position"]
)

transformed_df = pd.DataFrame(
    transformed_data, columns=numeric_cols + list(categorical_cols)
)
transformed_df.head(10)

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.0453,-0.3050,0.3273,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.7277,1.1051,1.2625,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.6364,-1.5818,1.0754,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.3583,-1.5772,-0.6078,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.0232,-0.5652,-0.4208,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.5625,0.8524,-0.2338,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,0.9493,-0.8958,0.7014,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,-0.8976,-0.9915,1.2625,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8,-1.4886,-0.5806,-0.2338,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.7277,1.7442,0.7014,0.0,0.0,0.0,1.0,0.0,0.0,1.0


## 5. Pipeline di scikit-learn

Pipeline merangkai beberapa langkah preprocessing dan training jadi satu objek, dijalankan berurutan. Manfaat utamanya: eksekusi konsisten di data training maupun testing, kode lebih ringkas, memudahkan hyperparameter tuning, dan mendukung modularitas supaya kita bisa bereksperimen dengan teknik berbeda tanpa menulis ulang banyak kode.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Pisahkan fitur dan target (anggap kolom terakhir adalah target)
X = transformed_df.iloc[:, :-1]
y = transformed_df.iloc[:, -1]

# Split data SEBELUM transformasi apa pun, supaya tidak terjadi data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# Buat pipeline sederhana: imputasi lalu scaling
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)  # hanya transform, tidak fit ulang

X_train_transformed = pd.DataFrame(
    X_train_transformed, columns=X_train.columns, index=X_train.index
)
X_test_transformed = pd.DataFrame(
    X_test_transformed, columns=X_test.columns, index=X_test.index
)

X_train_transformed.head(10)

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
13,0.8343,-0.8675,-1.5598,-0.5774,-0.4804,2.0817,-0.7746,-0.5774,-1.0
12,-0.6892,1.2828,-0.1509,-0.5774,2.0817,-0.4804,-0.7746,-0.5774,1.0
16,0.5441,-1.1734,-1.7610,-0.5774,-0.4804,2.0817,-0.7746,-0.5774,1.0
5,-1.3421,0.8215,-0.1509,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,-1.0
17,1.4147,0.6111,0.6541,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,1.0
2,-1.4147,-1.4798,1.2579,1.7321,-0.4804,-0.4804,-0.7746,-0.5774,1.0
10,1.2696,0.5768,0.4528,1.7321,-0.4804,-0.4804,-0.7746,1.7321,-1.0
3,0.5441,-1.4755,-0.5535,-0.5774,-0.4804,2.0817,-0.7746,1.7321,-1.0
1,0.9068,1.0604,1.4591,-0.5774,-0.4804,-0.4804,1.2910,1.7321,-1.0
9,0.9068,1.6647,0.8554,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,-1.0


Perhatikan urutannya: `train_test_split()` dipanggil **sebelum** transformasi apa pun diterapkan. Ini penting untuk mencegah data leakage, di mana informasi dari data testing ikut memengaruhi parameter transformasi (seperti mean dan standar deviasi) yang seharusnya hanya dipelajari dari data training. Setelah pipeline di-fit pada data training, kita hanya memanggil `transform()` (bukan `fit()` lagi) pada data testing, supaya parameter transformasinya tetap konsisten dan mensimulasikan skenario dunia nyata: data baru yang belum pernah dilihat model.

### Visualisasi Pipeline

scikit-learn juga menyediakan visualisasi pipeline dalam bentuk diagram lewat `set_config(display="diagram")`, yang membantu memahami alur data melalui tiap tahap transformasi.

In [15]:
from sklearn import set_config

set_config(display="diagram")
pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

## 6. Feature Engineering

Feature engineering mencakup dua aktivitas utama: **feature extraction** (membuat fitur baru dari fitur yang sudah ada) dan **feature selection** (memilih fitur yang paling relevan, membuang yang tidak berguna). Tujuannya: model yang lebih sederhana, generalisasi lebih baik, dan mengurangi risiko overfitting.

### PolynomialFeatures()

Membuat fitur polynomial dan interaksi dari fitur numerik yang sudah ada. Berguna untuk menangkap hubungan non-linear antar fitur tanpa mengubah algoritma modelnya sendiri.

In [16]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features, columns=poly.get_feature_names_out(X_train_transformed.columns)
)
poly_features_df.head(10)

,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.8343,-0.8675,-1.5598,-0.5774,-0.4804,2.0817,-0.7746,-0.5774,-1.0,...,4.3333,-1.6125,-1.2019,-2.0817,0.6000,0.4472,0.7746,0.3333,0.5774,1.0
1,1.0,-0.6892,1.2828,-0.1509,-0.5774,2.0817,-0.4804,-0.7746,-0.5774,1.0,...,0.2308,0.3721,0.2774,-0.4804,0.6000,0.4472,-0.7746,0.3333,-0.5774,1.0
2,1.0,0.5441,-1.1734,-1.7610,-0.5774,-0.4804,2.0817,-0.7746,-0.5774,1.0,...,4.3333,-1.6125,-1.2019,2.0817,0.6000,0.4472,-0.7746,0.3333,-0.5774,1.0
3,1.0,-1.3421,0.8215,-0.1509,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,-1.0,...,0.2308,-0.6202,0.2774,0.4804,1.6667,-0.7454,-1.2910,0.3333,0.5774,1.0
4,1.0,1.4147,0.6111,0.6541,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,1.0,...,0.2308,-0.6202,0.2774,-0.4804,1.6667,-0.7454,1.2910,0.3333,-0.5774,1.0
5,1.0,-1.4147,-1.4798,1.2579,1.7321,-0.4804,-0.4804,-0.7746,-0.5774,1.0,...,0.2308,0.3721,0.2774,-0.4804,0.6000,0.4472,-0.7746,0.3333,-0.5774,1.0
6,1.0,1.2696,0.5768,0.4528,1.7321,-0.4804,-0.4804,-0.7746,1.7321,-1.0,...,0.2308,0.3721,-0.8321,0.4804,0.6000,-1.3416,0.7746,3.0000,-1.7321,1.0
7,1.0,0.5441,-1.4755,-0.5535,-0.5774,-0.4804,2.0817,-0.7746,1.7321,-1.0,...,4.3333,-1.6125,3.6056,-2.0817,0.6000,-1.3416,0.7746,3.0000,-1.7321,1.0
8,1.0,0.9068,1.0604,1.4591,-0.5774,-0.4804,-0.4804,1.2910,1.7321,-1.0,...,0.2308,-0.6202,-0.8321,0.4804,1.6667,2.2361,-1.2910,3.0000,-1.7321,1.0
9,1.0,0.9068,1.6647,0.8554,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,-1.0,...,0.2308,-0.6202,0.2774,0.4804,1.6667,-0.7454,-1.2910,0.3333,0.5774,1.0


### KBinsDiscretizer()

Mengubah variabel kontinu jadi kategorikal dengan membaginya ke sejumlah "bin" atau kelompok diskrit. Jumlah bin diatur lewat `n_bins`, sementara `strategy` menentukan cara pembagiannya: `uniform` (lebar tiap bin sama), `quantile` (jumlah data di tiap bin sama), atau `kmeans` (berdasarkan jarak ke titik pusat kluster).

In [17]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)
binned_df.head(10)

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0
5,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0
6,2.0,1.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0
7,2.0,0.0,1.0,0.0,0.0,2.0,0.0,2.0,0.0
8,2.0,2.0,2.0,0.0,0.0,0.0,2.0,2.0,0.0
9,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0


### Seleksi Fitur: RFE() dan SelectFromModel()

Model yang ideal adalah yang berperforma baik dengan jumlah fitur paling sedikit. **Recursive Feature Elimination (RFE)** bekerja dengan melatih model memakai semua fitur, lalu secara bertahap membuang fitur yang paling tidak penting berdasarkan skor dari estimator (misalnya linear regression), dan mengulangi proses ini sampai jumlah fitur yang diinginkan tersisa.

In [18]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)
rfe.fit(X_train_transformed, y_train)

print("Ranking fitur (1 = paling penting):")
for nama_fitur, ranking in zip(X_train_transformed.columns, rfe.ranking_):
    print(f"  {nama_fitur}: {ranking}")

Ranking fitur (1 = paling penting):
  Age_scaled: 5
  Salary_scaled: 3
  Experience_scaled: 8
  Department_Finance: 7
  Department_HR: 6
  Department_IT: 4
  Department_Sales: 9
  Position_Junior: 2
  Position_Manager: 1


`SelectFromModel()` memilih fitur berdasarkan bobot pentingnya dari sebuah model, dengan ambang batas (`threshold`) tertentu, misalnya rata-rata dari semua skor pentingnya. Pendekatan ini sangat berguna dipadukan dengan model berbasis tree seperti Random Forest atau gradient boosting.

In [19]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=LinearRegression(), prefit=False, threshold="mean"
)
selector.fit(X_train_transformed, y_train)

selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

feature_importance = pd.DataFrame({
    "Feature": X_train_transformed.columns,
    "Importance": selector.estimator_.coef_,
    "Selected": selected_features_mask,
})
feature_importance.sort_values("Importance", key=abs, ascending=False)

,Feature,Importance,Selected
8,Position_Manager,-5.0000e-01,True
7,Position_Junior,-4.3301e-01,True
2,Experience_scaled,-1.1918e-15,False
1,Salary_scaled,9.2981e-16,False
0,Age_scaled,6.7550e-16,False
3,Department_Finance,5.3429e-16,False
4,Department_HR,-2.9837e-16,False
5,Department_IT,-2.8103e-16,False
6,Department_Sales,-1.5613e-16,False


Tabel di atas menunjukkan skor kepentingan tiap fitur sekaligus status apakah fitur itu terpilih untuk dipertahankan dalam model, berdasarkan ambang batas yang kita tentukan.

## 7. Latihan Komprehensif: California Housing Dataset

Sebagai penutup bab, buku mengajak membangun pipeline lengkap memakai dataset California Housing (20.640 baris, target berupa harga rumah rata-rata). Buku memuat dataset ini lewat `fetch_california_housing()` bawaan scikit-learn, tapi di lingkungan notebook ini pengunduhannya diblokir firewall sandbox, jadi kita ambil dataset yang persis sama lewat salinan CSV publik di GitHub. Isi datanya identik: 10 kolom, termasuk satu kolom kategorikal (`ocean_proximity`) dan sedikit missing value di `total_bedrooms`, yang justru pas untuk mempraktikkan seluruh teknik di bab ini sekaligus.

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Muat dataset California Housing
housing_url = (
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
    "datasets/housing/housing.csv"
)
housing_df = pd.read_csv(housing_url)

X_housing = housing_df.drop(columns=["median_house_value"])
y_housing = housing_df["median_house_value"]

print(f"Jumlah baris: {X_housing.shape[0]}, jumlah fitur: {X_housing.shape[1]}")
X_housing.head()

Jumlah baris: 20640, jumlah fitur: 9


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY


In [21]:
# 2. Split data
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=2024
)

# Pisahkan nama kolom numerik dan kategorikal untuk ColumnTransformer
numeric_features = X_housing.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_housing.select_dtypes(exclude=[np.number]).columns.tolist()
print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

# 3. Pipeline komprehensif: imputasi + scaling untuk kolom numerik,
#    one-hot encoding untuk kolom kategorikal, lalu Random Forest sebagai estimator akhir
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

housing_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=2024, n_jobs=-1)),
])

# 4. Fit pipeline ke data training
housing_pipeline.fit(X_train_h, y_train_h)

Fitur numerik: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Fitur kategorikal: ['ocean_proximity']


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['longitude', 'latitude',
                                                   'housing_median_age',
                                                   'total_rooms',
                                                   'total_bedrooms',
                                                   'population', 'households',
                                                   'median_income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ocean_proximity'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=2024))])

In [22]:
# 5. Evaluasi performa di data testing
y_pred_housing = housing_pipeline.predict(X_test_h)

mse_housing = mean_squared_error(y_test_h, y_pred_housing)
r2_housing = r2_score(y_test_h, y_pred_housing)

print(f"Mean Squared Error: {mse_housing:.4f}")
print(f"R-squared: {r2_housing:.4f}")

Mean Squared Error: 2419559801.3992
R-squared: 0.8231


Pipeline ini merangkum keseluruhan alur bab ini: split data lebih dulu untuk mencegah data leakage, scaling fitur numerik, lalu melatih model, semuanya dalam satu objek `Pipeline()` yang bisa dipakai ulang secara konsisten baik untuk training maupun saat model dipakai di data baru nanti.

## Poin-poin Penting Bab 2

- Kualitas data mentah menentukan performa model lebih dari sekadar kompleksitas algoritma yang dipakai, "garbage in, garbage out" benar-benar berlaku di ML.
- Ada tiga pendekatan imputasi utama: `SimpleImputer()` (cepat, ringan), `KNNImputer()` (menangkap hubungan antar fitur numerik), dan `IterativeImputer()` (paling akurat tapi paling berat secara komputasi).
- Scaling penting untuk algoritma berbasis jarak (KNN, SVM) dan algoritma berbasis gradient (linear regression), tapi tidak dibutuhkan oleh model berbasis tree.
- Encoding variabel kategorikal harus disesuaikan dengan jenisnya: `OneHotEncoder()` untuk nominal, `LabelEncoder()` dengan hati-hati untuk ordinal, dan `ColumnTransformer()` untuk menangani campuran tipe data sekaligus.
- `Pipeline()` merangkai semua langkah preprocessing dan training jadi satu objek yang konsisten, dan split data harus dilakukan sebelum transformasi apa pun untuk mencegah data leakage.
- Feature engineering mencakup pembuatan fitur baru (`PolynomialFeatures()`, `KBinsDiscretizer()`) dan seleksi fitur (`RFE()`, `SelectFromModel()`) untuk menyederhanakan model tanpa kehilangan informasi penting.

Referensi: Sukup, J. (2025). *scikit-learn Cookbook*, Third Edition, Bab 2 "Pre-Model Workflow and Data Preprocessing". Packt Publishing.